# Práctica 5 — SVM en Dataset Pima Indians Diabetes
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Aplicar SVM con diferentes kernels al dataset de diabetes, evaluar el impacto de la normalización y optimizar con GridSearchCV.

**Dataset:** Pima Indians Diabetes — 768 pacientes, 8 variables clínicas, clasificación binaria (diabetes sí/no)

⚠️ **Instrucciones:**
- Celdas marcadas con `# 🔧 TU CÓDIGO` debes completarlas.
- Responde las preguntas ❓ en celdas Markdown nuevas.
- Guarda una copia en Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     ConfusionMatrixDisplay, roc_curve, roc_auc_score)
from sklearn.pipeline        import Pipeline

print('✅ Librerías cargadas')

## Parte 1 — Carga y Exploración del Dataset

In [ ]:
# Cargar el dataset
columnas = ['embarazos', 'glucosa', 'presion', 'pliegue', 'insulina',
            'imc', 'diabetes_pedigree', 'edad', 'resultado']
df = pd.read_csv(
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    names=columnas
)
print(f'Forma: {df.shape}')
print(f'\nPrimeras filas:')
display(df.head())
print(f'\nClases: {df["resultado"].value_counts().to_dict()}')

In [ ]:
# 🔧 TU CÓDIGO
# 1. Separa las features (X) de la variable objetivo (y)
#    X = todas las columnas excepto 'resultado'
#    y = columna 'resultado'
# 2. Imprime el shape de X y los valores únicos de y

X = ___________
y = ___________

print(f'X shape: {X.shape}')
print(f'y valores únicos: {np.unique(y, return_counts=True)}')

In [ ]:
# 🔧 TU CÓDIGO
# Crea histogramas de las 8 features separados por clase (0 vs 1)
# Usa plt.subplots(2, 4, figsize=(14, 7))
# Para cada feature: histograma de los no diabéticos (azul) y diabéticos (rojo)
# ¿Qué features tienen distribuciones más distintas entre clases?

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
features = columnas[:-1]

for ax, feat in zip(axes.flat, features):
    # ___ tu código aquí ___
    pass

plt.suptitle('Distribución de features por clase — Pima Diabetes', fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 1
*(Responde aquí en Markdown)*

1. ¿Qué feature tiene la distribución más diferente entre clases 0 y 1?
2. ¿Hay valores de 0 en columnas como glucosa o presión? ¿Son datos reales o valores faltantes?

## Parte 2 — Impacto de la Normalización en SVM

In [ ]:
# 🔧 TU CÓDIGO
# 1. Divide X e y: 70/30, random_state=42, stratify=y
# 2. Entrena SVC(kernel='rbf', C=1, gamma='scale') SIN escalar
# 3. Repite CON StandardScaler
# 4. Imprime tabla comparativa (test accuracy y CV-10)

X_train, X_test, y_train, y_test = ___________

# Sin normalizar
svm_sin = SVC(kernel='rbf', C=1, gamma='scale')
# ___ entrenar ___

# Con normalizar
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)
svm_con = SVC(kernel='rbf', C=1, gamma='scale')
# ___ entrenar ___

print('── Comparativa Normalización ─────────────────────────────')
print(f'{"":20s} {"Sin scaler":>12s} {"Con scaler":>12s}')
# ___ imprime test accuracy y CV-10 ___

In [ ]:
# 🔧 TU CÓDIGO
# Con datos normalizados, compara kernels: 'linear', 'rbf', 'poly', 'sigmoid'
# Reporta test accuracy y CV-10 para cada uno

kernels = ['linear', 'rbf', 'poly', 'sigmoid']

for kern in kernels:
    # ___ tu código aquí ___
    pass

### ❓ Preguntas Parte 2
*(Responde aquí en Markdown)*

1. ¿Cuánto mejora la accuracy al normalizar? ¿Por qué?
2. ¿Qué kernel da mejor resultado en Pima Diabetes?

## Parte 3 — GridSearchCV: Mejores Parámetros

In [ ]:
# 🔧 TU CÓDIGO
# Pipeline([('scaler', StandardScaler()), ('svm', SVC(probability=True))])
# param_grid con svm__kernel, svm__C, svm__gamma
# GridSearchCV con cv=5, scoring='f1'

pipe = Pipeline([('scaler', ___________), ('svm', ___________)])
param_grid = {
    'svm__kernel': ___________,
    'svm__C':      ___________,
    'svm__gamma':  ___________,
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Mejores params: {grid.best_params_}')
print(f'CV-5 F1:        {grid.best_score_:.4f}')
print(f'Test accuracy:  {grid.score(X_test, y_test):.4f}')

In [ ]:
# 🔧 TU CÓDIGO
# Predice con el mejor modelo
# Imprime classification_report
# Muestra ConfusionMatrixDisplay

y_pred_best = grid.predict(X_test)
print('── Classification Report ─────────────────────────────────')
print(classification_report(y_test, y_pred_best, target_names=['No Diabética', 'Diabética']))

fig, ax = plt.subplots(figsize=(5, 4))
# ___ ConfusionMatrixDisplay ___
plt.tight_layout()
plt.show()

### ❓ Preguntas Parte 3
*(Responde aquí en Markdown)*

1. ¿Qué es más grave: falso positivo o falso negativo en diagnóstico de diabetes?
2. ¿El modelo tiene mejor Precision o Recall para la clase 'Diabética'?
3. ¿Por qué usamos F1 en GridSearchCV en lugar de accuracy?

## Parte 4 — Curva ROC del Mejor SVM

In [ ]:
# 🔧 TU CÓDIGO
# 1. Obtén probabilidades con grid.predict_proba(X_test)
#    (El pipeline necesita SVC(probability=True) — ajusta si es necesario)
# 2. Calcula fpr, tpr con roc_curve(y_test, proba[:, 1])
# 3. Calcula AUC con roc_auc_score
# 4. Grafica la curva ROC

# ___ tu código aquí ___

### ❓ Preguntas Parte 4
*(Responde aquí en Markdown)*

1. ¿El AUC es un buen indicador para diagnóstico clínico? (AUC > 0.80 = bueno)
2. ¿En qué punto del umbral operarías: máximo recall o punto de equilibrio?

## 🌟 Desafío Bonus — Limpieza de datos (Opcional)

In [ ]:
# 🔧 DESAFÍO
# Los 0s en glucosa, presion, pliegue, insulina, imc son datos faltantes enmascarados.
# 1. Reemplaza 0s por NaN en esas columnas
# 2. Imputa con la mediana de cada columna
# 3. Reentrena el mejor SVM con datos limpios
# 4. ¿Mejora el F1?

cols_con_ceros = ['glucosa', 'presion', 'pliegue', 'insulina', 'imc']
# ___ tu código aquí ___

## 📝 Conclusiones

*(Escribe aquí tu párrafo de conclusiones)*

Responde: ¿Cuál fue el mejor kernel y por qué? ¿Cuánto mejoró la normalización? ¿Priorizarías Recall o Precision para diagnóstico de diabetes? Justifica con los resultados obtenidos.